# Parte 3: Mecanismos de Verificacion y Control de Integridad
## 03_Tarea_Mecanismos_Verificacion.ipynb

Esta libreta implementa los mecanismos tecnicos de verificacion de integridad sobre datasets reales:

- **Ejercicio 1**: Validacion de frontera (Schema Boundary) usando el dataset **UCI Air Quality** (id=360) - filtramos antes de que el dato entre a la zona analitica.
- **Ejercicio 2**: Checksums MD5 para detectar corrupcion de contenido, simulando envio de paquetes de datos entre sistemas.
- **Ejercicio 3**: Auditoria de reconciliacion multi-origen usando el **dataset de titulos de credito TMDB** (publico en GitHub) para cruzar dos fuentes y detectar discrepancias.

In [6]:
!pip install ucimlrepo -q
import pandas as pd
import numpy as np
import hashlib
from ucimlrepo import fetch_ucirepo

# Cargamos el dataset UCI Air Quality como fuente principal
air_quality = fetch_ucirepo(id=360)
df_aq = air_quality.data.features.copy()
df_aq = df_aq.replace(-200, np.nan)
print('UCI Air Quality cargado:', df_aq.shape)
df_aq.head()

UCI Air Quality cargado: (9357, 15)


,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,3/10/2004,18:00:00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578
1,3/10/2004,19:00:00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255
2,3/10/2004,20:00:00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502
3,3/10/2004,21:00:00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867
4,3/10/2004,22:00:00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888


---
## Ejercicio 1: Validacion de Frontera (Schema Boundary) y Cuarentena

### Contexto teorico
La **validacion de frontera** (o Schema Boundary Validation) es el mecanismo por el cual se establecen filtros estrictos en el punto de entrada a la zona analitica. Actua como una 'aduana' de datos: ningun registro entra a la zona Silver/Gold sin haber superado las reglas del contrato de datos.

Este mecanismo es especialmente importante en entornos distribuidos donde multiples productores con diferentes versiones de firmware pueden enviar datos con esquemas distintos.

El contrato de datos para UCI Air Quality define:
- `Date`: no nulo, formato DD/MM/YYYY
- `Time`: no nulo, formato HH.MM.SS
- `T` (temperatura): numerica, rango [-30, 70]
- `RH` (humedad): numerica, rango [0, 100]
- `CO(GT)`: numerica, >= 0

Registros que no cumplen el contrato van a zona de **cuarentena** con el motivo de rechazo etiquetado.

In [7]:
# === SCHEMA BOUNDARY VALIDATION ===
# El contrato de datos define las reglas que TODO registro debe cumplir
# antes de entrar a la zona analitica (Silver/Gold layer).
# Si un registro no supera UNA SOLA regla -> cuarentena con motivo etiquetado.
# Esto es una 'aduana': ningun dato malformado entra a la zona de analisis.

def validate_row(row):
    motivos = []

    # Regla 1: identificadores temporales no nulos
    # Sin Date o Time el registro no puede ubicarse en ninguna serie temporal
    # -> cuarentena inmediata independientemente del resto de campos
    if pd.isnull(row.get('Date', None)): motivos.append('Date_NULL')
    if pd.isnull(row.get('Time', None)): motivos.append('Time_NULL')

    # Regla 2: temperatura en rango fisico [-30C, 70C]
    # Fuera de este rango en entorno urbano europeo -> fallo del sensor
    if 'T' in row and not pd.isnull(row['T']):
        if not (-30 <= row['T'] <= 70): motivos.append(f'T_OUT_RANGE({row["T"]:.1f})')

    # Regla 3: humedad relativa en rango fisico [0%, 100%]
    # Limite absoluto termodinamico: no existe humedad negativa ni mayor al 100%
    if 'RH' in row and not pd.isnull(row['RH']):
        if not (0 <= row['RH'] <= 100): motivos.append(f'RH_OUT_RANGE({row["RH"]:.1f})')

    # Regla 4: concentracion de CO no negativa [>= 0 mg/m3]
    # Fisicamente imposible tener concentracion negativa de un gas.
    # Esta regla estaba en el contrato teorico pero faltaba en el codigo original.
    if 'CO(GT)' in row and not pd.isnull(row['CO(GT)']):
        if row['CO(GT)'] < 0: motivos.append(f'CO_GT_NEGATIVE({row["CO(GT)"]:.2f})')

    # Regla 5: payload de medicion no completamente vacio
    # Si TODOS los campos de sensor son NaN, el registro llego vacio -> cuarentena.
    # Un registro solo con Date/Time pero sin ninguna medicion no aporta valor analitico.
    sensor_cols = ['CO(GT)', 'PT08.S1(CO)', 'NMHC(GT)', 'C6H6(GT)', 'T', 'RH', 'AH']
    sensor_vals = [row.get(c) for c in sensor_cols if c in row]
    if sensor_vals and all(pd.isnull(v) for v in sensor_vals):
        motivos.append('PAYLOAD_VACIO')

    return motivos

# Aplicamos el contrato a todo el dataset
df_aq['_violations'] = df_aq.apply(validate_row, axis=1)
df_aq['_valid'] = df_aq['_violations'].apply(lambda x: len(x) == 0)

df_zone_analytic = df_aq[df_aq['_valid']].drop(columns=['_violations','_valid']).copy().reset_index(drop=True)
df_cuarentena    = df_aq[~df_aq['_valid']].copy()

print(f'Registros que pasan frontera (zona analitica): {len(df_zone_analytic)}')
print(f'Registros en cuarentena: {len(df_cuarentena)}')
print()
print('Distribucion de motivos de rechazo en cuarentena:')
print(df_cuarentena['_violations'].explode().value_counts().head(10))

Registros que pasan frontera (zona analitica): 9321
Registros en cuarentena: 36

Distribucion de motivos de rechazo en cuarentena:
_violations
PAYLOAD_VACIO    36
Name: count, dtype: int64


---
## Ejercicio 2: Hashes/Checksums para Corrupcion de Contenido

### Contexto teorico
El uso de **checksums (MD5, SHA-256)** como mecanismo de verificacion permite detectar si un bloque de datos ha sido modificado o corrompido durante el transito entre sistemas (red, almacenamiento, procesado).

El flujo tipico es:
1. El productor calcula el hash del payload antes de enviarlo y lo incluye en la cabecera del mensaje.
2. El consumidor recalcula el hash del payload recibido y lo compara con el de la cabecera.
3. Si no coinciden -> el paquete fue corrompido en transito y se descarta (o se solicita reenvio).

Usamos MD5 (y no SHA-256) por su velocidad en escenarios de alta frecuencia IoT donde la latencia importa. En escenarios de seguridad critica se usaria SHA-256 o SHA-3.

In [8]:
import hashlib

# Simulamos el envio de bloques de 100 filas del dataset (como si fuesen paquetes de red)
# El productor calcula el MD5 de cada bloque antes de enviarlo
df_send = df_zone_analytic.head(500).copy()

def compute_md5(df_block):
    content = df_block.to_csv(index=False).encode('utf-8')
    return hashlib.md5(content).hexdigest()

# Partimos en bloques de 100 filas (simulando batches de ingestion)
blocks = [df_send.iloc[i:i+100].copy() for i in range(0, 500, 100)]
hashes_originales = [compute_md5(b) for b in blocks]

print('Hashes MD5 calculados por el productor (bloques de 100 filas):')
for i, h in enumerate(hashes_originales):
    print(f'  Bloque {i}: {h}')

Hashes MD5 calculados por el productor (bloques de 100 filas):
  Bloque 0: 792ea767ee46426783a8d6b606cd452d
  Bloque 1: 9a3a3a91e61c1fc8a423a6b9b9265916
  Bloque 2: 547e06cdb3f80f11792f83646c629634
  Bloque 3: 3b488c8604b22ffd98911b097417e15d
  Bloque 4: c122482290b80049578eb55eeea9663e


In [9]:
# Simulamos corrupcion en transito: modificamos un valor en el bloque 2
# Esto replica lo que ocurre cuando un bit flip en la red o un disco defectuoso
# altera un byte del payload sin que el protocolo de transporte lo detecte.
blocks_recibidos = [b.copy() for b in blocks]
if 'T' in blocks_recibidos[2].columns:
    blocks_recibidos[2].iloc[5, blocks_recibidos[2].columns.get_loc('T')] = 9999.0  # bit-flip simulado
else:
    num_col = blocks_recibidos[2].select_dtypes(include=np.number).columns[0]
    blocks_recibidos[2].iloc[5, blocks_recibidos[2].columns.get_loc(num_col)] = 9999.0

# === PASO 1: VERIFICACION DE INTEGRIDAD EN EL CONSUMIDOR ===
# El consumidor recalcula el hash de cada bloque recibido y lo compara
# con el hash que envio el productor en la cabecera del mensaje.
# Si no coinciden -> el contenido fue modificado en transito: descartamos el bloque.
print('=== VERIFICACION DE INTEGRIDAD EN EL CONSUMIDOR ===')
bloques_ok = []
bloques_corruptos = []

for i, (b_orig, b_recv) in enumerate(zip(blocks, blocks_recibidos)):
    h_orig = hashes_originales[i]
    h_recv = compute_md5(b_recv)
    if h_orig == h_recv:
        status = 'OK'
        bloques_ok.append(i)
    else:
        status = '*** CORRUPCION DETECTADA ***'
        bloques_corruptos.append(i)
    print(f'Bloque {i}: original={h_orig[:12]}... recibido={h_recv[:12]}... -> {status}')

# === PASO 2: ACCION POST-DETECCION ===
# Los bloques corruptos NO se procesan ni se imputan: se mandan a cuarentena
# y se emite una solicitud de reenvio al productor.
# Razon: no sabemos QUE bytes cambiaron -> cualquier imputacion seria arbitraria
# y corromperia el analisis. La unica accion segura es pedir el bloque original.
print()
print('=== ACCION POST-DETECCION ===')
print(f'Bloques integros aceptados para procesado:  {bloques_ok}')
print(f'Bloques corruptos en cuarentena (reenvio):  {bloques_corruptos}')

# Cuarentena: concatenamos los bloques corruptos recibidos para auditoria
if bloques_corruptos:
    df_bloques_corruptos = pd.concat([blocks_recibidos[i] for i in bloques_corruptos], ignore_index=True)
    print(f'Filas en cuarentena de corrupcion de transito: {len(df_bloques_corruptos)}')
    print(f'Accion: solicitar reenvio de bloques {bloques_corruptos} al productor')

# Dataset integro: solo los bloques que pasaron la verificacion
df_integro = pd.concat([blocks_recibidos[i] for i in bloques_ok], ignore_index=True)
print(f'Filas validas listas para ingestion: {len(df_integro)}')
df_integro.head()

=== VERIFICACION DE INTEGRIDAD EN EL CONSUMIDOR ===
Bloque 0: original=792ea767ee46... recibido=792ea767ee46... -> OK
Bloque 1: original=9a3a3a91e61c... recibido=9a3a3a91e61c... -> OK
Bloque 2: original=547e06cdb3f8... recibido=fb10cc9ff3f7... -> *** CORRUPCION DETECTADA ***
Bloque 3: original=3b488c8604b2... recibido=3b488c8604b2... -> OK
Bloque 4: original=c122482290b8... recibido=c122482290b8... -> OK

=== ACCION POST-DETECCION ===
Bloques integros aceptados para procesado:  [0, 1, 3, 4]
Bloques corruptos en cuarentena (reenvio):  [2]
Filas en cuarentena de corrupcion de transito: 100
Accion: solicitar reenvio de bloques [2] al productor
Filas validas listas para ingestion: 400


,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,3/10/2004,18:00:00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578
1,3/10/2004,19:00:00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255
2,3/10/2004,20:00:00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502
3,3/10/2004,21:00:00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867
4,3/10/2004,22:00:00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888


---
## Ejercicio 3: Auditoria de Reconciliacion (Source of Truth)

### Contexto teorico
La **reconciliacion** es el proceso de cruzar datos procedentes de dos o mas fuentes independientes para detectar discrepancias y determinar cual es la 'fuente de verdad' (Source of Truth).

En entornos distribuidos, el mismo evento puede ser registrado por sistemas distintos (ej. base de datos interna vs pasarela bancaria externa, o sensor local vs servidor central). Las discrepancias pueden deberse a: retrasos de red, errores de redondeo, diferencias de zona horaria, o corrupciones silenciosas.

Usamos dos particiones del mismo dataset UCI Air Quality como simulacion de dos fuentes (la BD interna y la gateway externa), introduciendo discrepancias artificiales para demostrar la tecnica.

In [10]:
# === RECONCILIACION MULTI-ORIGEN ===
# Simulamos dos fuentes sobre los mismos 1000 registros del UCI Air Quality:
#   Fuente A: base de datos interna (sistema transaccional propio)
#   Fuente B: gateway externa (sistema bancario/externo que recibe los mismos datos)
#
# PROBLEMA con la version original: usaba 'Date' como clave de reconciliacion.
# Date NO es unica en este dataset (hay ~24 filas por dia), lo que produce un
# join N x N: cada fecha de A cruza con TODAS las filas de esa fecha en B.
# Resultado: 16833 discrepancias detectadas cuando solo habia 50 inyectadas.
#
# SOLUCION: la clave de reconciliacion debe ser un IDENTIFICADOR UNICO de registro.
# Usamos 'record_id' (indice posicional, equivalente al ID de transaccion en un
# sistema real). En produccion seria el UUID de la transaccion o el transaction_id.

np.random.seed(42)
cols_num = df_zone_analytic.select_dtypes(include=np.number).columns.tolist()
col_val  = cols_num[0] if cols_num else df_zone_analytic.columns[1]

# Creamos las fuentes con record_id como clave natural unica
df_base = df_zone_analytic.head(1000).copy().reset_index(drop=True)
df_base['record_id'] = df_base.index  # ID unico y estable por registro

# Fuente A: BD interna
df_fuente_A = df_base[['record_id', col_val]].copy()
df_fuente_A.columns = ['record_id', 'valor_A']

# Fuente B: gateway externa - mismos registros con ~5% de discrepancias
# Las discrepancias simulan errores de redondeo o retrasos de sincronizacion
df_fuente_B = df_fuente_A.copy()
df_fuente_B.columns = ['record_id', 'valor_B']
idx_disc = np.random.choice(1000, 50, replace=False)
df_fuente_B.loc[idx_disc, 'valor_B'] = df_fuente_B.loc[idx_disc, 'valor_B'] * 1.1 + 0.5

# Tambien simulamos 5 registros presentes solo en B (llegaron por la gateway pero
# no se registraron en la BD interna -> posible fallo de escritura en el sistema propio)
registros_solo_B = pd.DataFrame({
    'record_id': range(1000, 1005),
    'valor_B':   [1.5, 2.3, 0.8, 3.1, 1.9]
})
df_fuente_B = pd.concat([df_fuente_B, registros_solo_B], ignore_index=True)

# RECONCILIACION: cruzamos por record_id (clave unica) con outer join
# El outer join detecta tanto discrepancias de valor como registros huerfanos
# (presentes en una fuente pero no en la otra)
df_recon = pd.merge(df_fuente_A, df_fuente_B, on='record_id', how='outer', indicator=True)
df_recon['discrepancia'] = abs(df_recon['valor_A'] - df_recon['valor_B']) > 0.001

print(f'Registros en fuente A (BD interna):   {len(df_fuente_A)}')
print(f'Registros en fuente B (gateway ext):  {len(df_fuente_B)}')
print(f'Registros solo en A (ausentes en B):  {(df_recon["_merge"] == "left_only").sum()}')
print(f'Registros solo en B (ausentes en A):  {(df_recon["_merge"] == "right_only").sum()}')
print(f'Registros con discrepancia de valor:  {df_recon["discrepancia"].sum()}')
print(f'  (esperado ~50 inyectados + posible solape con only_B)')
print()
print('Muestra de discrepancias detectadas:')
df_recon[df_recon['discrepancia']].head()

Registros en fuente A (BD interna):   1000
Registros en fuente B (gateway ext):  1005
Registros solo en A (ausentes en B):  0
Registros solo en B (ausentes en A):  5
Registros con discrepancia de valor:  38
  (esperado ~50 inyectados + posible solape con only_B)

Muestra de discrepancias detectadas:


,record_id,valor_A,valor_B,_merge,discrepancia
59,59,1.0,1.60,both,True
76,76,3.1,3.91,both,True
96,96,2.5,3.25,both,True
101,101,2.2,2.92,both,True
136,136,5.3,6.33,both,True


---
## Resumen de mecanismos de verificacion

| Mecanismo | Que detecta | Cuando aplicar |
|-----------|------------|----------------|
| Schema Boundary | Datos malformados o fuera de contrato | En la frontera de ingestion, antes de la zona analitica |
| Checksum MD5 | Corrupcion de contenido en transito | En la recepcion de cada batch/paquete de datos |
| Reconciliacion | Discrepancias entre fuentes y registros huerfanos | En el cierre de periodo (diario/mensual) |

Los tres mecanismos son complementarios: el checksum actua primero (corrupcion binaria), luego el schema boundary (corrupcion semantica), y finalmente la reconciliacion (inconsistencias entre sistemas).